In [ ]:
import pandas as pd
import re
import numpy as np # For np.nan
import os # Added for path joining

def extract_and_convert_answer(raw_pred_text):
    """
    Extracts the answer from the raw_preds text.
    It prioritizes direct Arabic answers like [[ب]],
    then English answers like C., then English answers like [[C]].
    """
    if pd.isna(raw_pred_text): # Handle cases where raw_preds itself might be NaN
        return None

    raw_pred_text_str = str(raw_pred_text)

    # 1. Check for direct Arabic answer in double square brackets: e.g., "[[ب]]"
    # The regex looks for "Final Answer: The final answer is [[(one of أ, ب, ج, د)]]".
    arabic_letters = "أ|ب|ج|د" # Pipe means OR
    match_direct_arabic = re.search(r"Final Answer: The final answer is \[\[(" + arabic_letters + r")\]\]", raw_pred_text_str)
    if match_direct_arabic:
        return match_direct_arabic.group(1) # Return the matched Arabic letter directly

    # 2. Check for English answer (A, B, C, D) followed by a period: e.g., "C."
    # The regex looks for "Final Answer: The final answer is X." where X is a single uppercase English letter.
    match_english_period = re.search(r"Final Answer: The final answer is ([A-D])\.", raw_pred_text_str)
    if match_english_period:
        english_answer = match_english_period.group(1)
        answer_map = {'A': 'أ', 'B': 'ب', 'C': 'ج', 'D': 'د'}
        return answer_map.get(english_answer)

    # 3. Fallback: Check for English answer (A, B, C, D) in double square brackets: e.g., "[[C]]"
    match_english_brackets = re.search(r"Final Answer: The final answer is \[\[([A-D])\]\]", raw_pred_text_str)
    if match_english_brackets:
        english_answer = match_english_brackets.group(1)
        answer_map = {'A': 'أ', 'B': 'ب', 'C': 'ج', 'D': 'د'}
        return answer_map.get(english_answer)

    return None # Return None if no pattern is found

def process_csv(input_filepath, output_filepath):
    """
    Reads a CSV, processes the 'preds' column, and saves the result.
    """
    try:
        # Read the CSV file
        df = pd.read_csv(input_filepath)
    except FileNotFoundError:
        print(f"Error: The file {input_filepath} was not found.")
        print("Please ensure the 'data' folder exists and contains the input CSV.")
        return
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return

    # Ensure the necessary columns exist
    required_columns = ['preds', 'raw_preds']
    for col in required_columns:
        if col not in df.columns:
            print(f"Error: Required column '{col}' not found in the CSV.")
            return

    # Iterate over rows and fill 'preds' where it's null or empty
    for index, row in df.iterrows():
        # Check if 'preds' is NaN (pandas' way of saying null) or an empty string
        if pd.isna(row['preds']) or str(row['preds']).strip() == "":
            arabic_answer = extract_and_convert_answer(row['raw_preds'])
            if arabic_answer:
                df.loc[index, 'preds'] = arabic_answer
                # print(f"Row {index}: Filled 'preds' with '{arabic_answer}' from raw_preds: '{row['raw_preds'][:100]}...'") # Optional: for logging

    try:
        # Ensure the output directory exists
        output_dir = os.path.dirname(output_filepath)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"Created directory: {output_dir}")

        # Save the modified DataFrame to a new CSV file
        df.to_csv(output_filepath, index=False, encoding='utf-8-sig') # utf-8-sig for better Excel compatibility with Arabic
        print(f"Processing complete. Output saved to {output_filepath}")
    except Exception as e:
        print(f"Error writing CSV file: {e}")

# --- Main execution ---
if __name__ == "__main__":
    # Define the data subfolder
    data_folder = "data"

    # Define input and output file paths within the data folder
    input_csv_file = os.path.join(data_folder, 'input.csv')
    output_csv_file = os.path.join(data_folder, 'output_filled.csv')

    # Create the data directory if it doesn't exist
    if not os.path.exists(data_folder):
        try:
            os.makedirs(data_folder)
            print(f"Created directory: {data_folder}")
        except OSError as e:
            print(f"Error creating directory {data_folder}: {e}")
            exit()


    # Create a dummy input.csv for demonstration if it doesn't exist
    # This dummy will now reflect all supported raw_preds formats
    try:
        with open(input_csv_file, 'r', encoding='utf-8-sig') as f:
            pass # File exists
    except FileNotFoundError:
        print(f"'{input_csv_file}' not found. Creating a dummy file for demonstration in '{data_folder}/'.")
        dummy_data = {
            'golds': [0, 1, 0, 1, 0],
            'options': [
                "['Option Set 1']",
                "['Option Set 2']",
                "['Option Set 3']",
                "['Option Set 4']",
                "['Option Set 5']"
            ],
            'preds': [None, 'أ', '', None, None], # One NaN, one filled, one empty string, two Nones for new tests
            'raw_preds': [
                """**1. Deconstruct... **6. Synthesize and Decide:**
- **Final Answer:** The final answer is C.""", # English with period
                "Some other raw prediction text... Final Answer: The final answer is A.", # Already filled, won't be overwritten
                """**1. Deconstruct... **6. Synthesize and Decide:**
Final Answer: The final answer is [[ب]].""", # Direct Arabic in brackets
                "Fallback test... Final Answer: The final answer is [[B]]", # English in brackets
                "Another test... Final Answer: The final answer is D." # English with period, different letter
            ],
            'subject': ['Subject A', 'Subject B', 'Subject C', 'Subject D', 'Subject E'],
            'ABILITY': ['Ability V', 'Ability W', 'Ability X', 'Ability Y', 'Ability Z'],
            'index': [1, 2, 3, 4, 5]
        }
        # Simplified dummy 'options' for brevity
        dummy_data['options'][0] = "['مينغكاي وجيانيينج كان لديهما مزحة حول اختيار أعضاء الفريق قبل ذلك، مينغكاي يفكر في تلك المزحة.', 'مينغكاي يبتسم لأنه يعتقد أن تاو تاو هو أفضل لاعب موجود.', 'مينغكاي يبتسم لجيانيينج لأنهم يعرفون أنهم لا يمكنهم اختيار سوى تاو تاو.', 'مينغكاي وجيانيينج لديهما خطة سرية حول تاو تاو، لذلك يبتسم.']"


        dummy_df = pd.DataFrame(dummy_data)
        try:
            dummy_df.to_csv(input_csv_file, index=False, encoding='utf-8-sig')
            print(f"Dummy '{input_csv_file}' created. Please replace it with your actual data.")
        except Exception as e:
            print(f"Error creating dummy file {input_csv_file}: {e}")

    process_csv(input_csv_file, output_csv_file)


Processing complete. Output saved to data/output_filled.csv
